# Coverage Audit

In [1]:
# Imports
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import re

In [2]:
WORKDIR = Path('/Users/ryanf/Documents/GitHub/benchmarking_dogwhistles')

@dataclass
class PipelineConfig:
    # `workdir` is the anchor used to resolve all relative paths in the notebook.
    workdir: Path = WORKDIR

    # Local sources expected to exist in the repository.
    data_local_path: Path = Path('outputs/preprocessing/dedup_primary.tsv')
    glossary_local_path: Path = Path('data/glossary.tsv')

    # Directory where notebook exports are written.
    output_dir: Path = Path('outputs/coverage_audits')


cfg = PipelineConfig()
# Convert all configured relative paths into absolute paths once up front.
cfg.output_dir = cfg.workdir / cfg.output_dir
cfg.data_path = cfg.workdir / cfg.data_local_path
cfg.glossary_path = cfg.workdir / cfg.glossary_local_path

# Create output/cache directories early so later cells can assume they exist.
cfg.output_dir.mkdir(parents=True, exist_ok=True)
cfg

PipelineConfig(workdir=WindowsPath('/Users/ryanf/Documents/GitHub/benchmarking_dogwhistles'), data_local_path=WindowsPath('outputs/preprocessing/dedup_primary.tsv'), glossary_local_path=WindowsPath('data/glossary.tsv'), output_dir=WindowsPath('/Users/ryanf/Documents/GitHub/benchmarking_dogwhistles/outputs/coverage_audits'))

In [3]:
# Load data
glossary = pd.read_csv(cfg.glossary_path, sep='\t')
data = pd.read_csv(cfg.data_path, sep='\t')

# Ensure everything is lowercase for matching
glossary['surface_form'] = glossary['surface_form'].str.strip().str.lower()
glossary['target'] = glossary['target'].str.strip().str.lower()

# Pre-calculate glossary totals for denominators
# This tells us how many terms exist for every Level/Target combination
glossary_stats = glossary.groupby(['taxonomy_level', 'target']).size().to_frame('total_terms')

# Compile regex: Sorting by length (descending) prevents partial matches 
# (e.g., matching 'dog' inside 'dogwhistle')
all_forms = sorted(glossary['surface_form'].unique(), key=len, reverse=True)
pattern = re.compile(r'\b(' + '|'.join(map(re.escape, all_forms)) + r')\b', flags=re.IGNORECASE)

In [4]:
# 1. Extract matches
data['found_forms'] = data['text'].apply(lambda x: pattern.findall(x.lower()) if pd.notna(x) else [])

# 2. Explode and Join
# Each row in 'matches_df' represents one instance of a found dogwhistle
matches_df = data.explode('found_forms').dropna(subset=['found_forms'])

# 3. Merge with glossary to get the taxonomy_level and target for each match
# This handles cases where one surface_form might belong to multiple categories
audit_df = matches_df.merge(
    glossary, 
    left_on='found_forms', 
    right_on='surface_form', 
    how='inner'
)

In [5]:
# Build metrics aggregated by taxonomy_level and target
metrics_data = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    # Get glossary reference: all forms and types for this level/target combination
    glossary_subset = glossary[(glossary['taxonomy_level'] == level) & 
                                (glossary['target'] == target)]
    total_glossary_forms = glossary_subset['surface_form'].nunique()
    total_glossary_types = glossary_subset['type'].nunique()
    
    # Calculate metrics
    # Presence rate: proportion of distinct surface forms found vs. glossary
    distinct_forms_found = group['found_forms'].nunique()
    presence_rate = distinct_forms_found / total_glossary_forms if total_glossary_forms > 0 else 0
    
    # Type coverage: proportion of distinct dogwhistle categories (types) represented
    # Use the 'type' column already present in the group (from earlier glossary merge)
    distinct_types_found = group['type'].nunique()
    type_coverage = distinct_types_found / total_glossary_types if total_glossary_types > 0 else 0
    
    # Token frequency: total count of matched instances
    total_tokens = len(group)
    
    metrics_data.append({
        'taxonomy_level': level,
        'target': target,
        'total_glossary_forms': total_glossary_forms,
        'total_glossary_types': total_glossary_types,
        'distinct_forms_found': distinct_forms_found,
        'distinct_types_found': distinct_types_found,
        'presence_rate': presence_rate,
        'type_coverage': type_coverage,
        'token_frequency': total_tokens
    })

final_report = pd.DataFrame(metrics_data)

In [6]:
final_report

,taxonomy_level,target,total_glossary_forms,total_glossary_types,distinct_forms_found,distinct_types_found,presence_rate,type_coverage,token_frequency
0,2,african,5,1,2,1,0.400000,1.0,2
1,3,african,1,1,1,1,1.000000,1.0,42
2,3,hispanic,3,1,1,1,0.333333,1.0,1
3,4,transgender women,4,1,2,1,0.500000,1.0,3


In [7]:
# Detailed breakdown: which forms appear in each level/target group
detailed_breakdown = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    forms_list = sorted(group['found_forms'].unique())
    for form in forms_list:
        form_count = len(group[group['found_forms'] == form])
        detailed_breakdown.append({
            'taxonomy_level': level,
            'target': target,
            'surface_form': form,
            'token_count': form_count
        })

detailed_report = pd.DataFrame(detailed_breakdown).sort_values(
    by=['taxonomy_level', 'target', 'token_count'], ascending=[True, True, False]
)

print("Detailed breakdown of surface forms found:")
print(detailed_report)

Detailed breakdown of surface forms found:
   taxonomy_level             target                surface_form  token_count
0               2            african             absentee father            1
1               2            african             lack of fathers            1
2               3            african          affirmative action           42
3               3           hispanic  end birthright citizenship            1
5               4  transgender women                actual women            2
4               4  transgender women                actual woman            1


In [8]:
# Summary: Missing forms (in glossary but not found in benchmark)
missing_forms = []

for (level, target), glossary_subset in glossary.groupby(['taxonomy_level', 'target']):
    glossary_forms = set(glossary_subset['surface_form'].unique())
    found_forms = set(detailed_report[(detailed_report['taxonomy_level'] == level) & 
                                      (detailed_report['target'] == target)]['surface_form'].unique())
    missing = glossary_forms - found_forms
    
    for form in sorted(missing):
        missing_forms.append({
            'taxonomy_level': level,
            'target': target,
            'surface_form': form,
            'status': 'not_found'
        })

missing_report = pd.DataFrame(missing_forms)

print("=" * 80)
print("AUDIT SUMMARY")
print("=" * 80)
print(f"\nTotal benchmark posts analyzed: {len(data)}")
print(f"Total dogwhistle matches found: {len(audit_df)}")
print(f"Unique surface forms found: {audit_df['found_forms'].nunique()}")
print(f"\nTotal surface forms in glossary: {len(glossary)}")
print(f"Surface forms NOT found in benchmark: {len(missing_report)}")

print("\n" + "=" * 80)
print("METRICS BY TAXONOMY LEVEL & TARGET GROUP")
print("=" * 80)
print(final_report.to_string(index=False))

if len(missing_report) > 0:
    print("\n" + "=" * 80)
    print("FORMS MISSING FROM BENCHMARK")
    print("=" * 80)
    print(missing_report.to_string(index=False))

AUDIT SUMMARY

Total benchmark posts analyzed: 59621
Total dogwhistle matches found: 48
Unique surface forms found: 6

Total surface forms in glossary: 13
Surface forms NOT found in benchmark: 7

METRICS BY TAXONOMY LEVEL & TARGET GROUP
 taxonomy_level            target  total_glossary_forms  total_glossary_types  distinct_forms_found  distinct_types_found  presence_rate  type_coverage  token_frequency
              2           african                     5                     1                     2                     1       0.400000            1.0                2
              3           african                     1                     1                     1                     1       1.000000            1.0               42
              3          hispanic                     3                     1                     1                     1       0.333333            1.0                1
              4 transgender women                     4                     1          

In [9]:
# Export results
audit_metrics_path = cfg.output_dir / 'audit_metrics.tsv'
audit_detailed_path = cfg.output_dir / 'audit_detailed.tsv'
audit_missing_path = cfg.output_dir / 'audit_missing.tsv'

final_report.to_csv(audit_metrics_path, sep='\t', index=False)
detailed_report.to_csv(audit_detailed_path, sep='\t', index=False)
if len(missing_report) > 0:
    missing_report.to_csv(audit_missing_path, sep='\t', index=False)

print(f"\nExported results:")
print(f"  - {audit_metrics_path}")
print(f"  - {audit_detailed_path}")
if len(missing_report) > 0:
    print(f"  - {audit_missing_path}")


Exported results:
  - \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\coverage_audits\audit_metrics.tsv
  - \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\coverage_audits\audit_detailed.tsv
  - \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\coverage_audits\audit_missing.tsv


## Annotation Quality Audit

For surface forms that appear in the benchmark, examine assigned labels. Decompose into:
- **Case A**: Present + labeled hateful (correct annotation)
- **Case B**: Present + labeled non-hateful (annotator failure)
- **Case C**: Absent from union (collection failure)

Metrics:
- **Correct Labeling Rate**: Case A / (Case A + Case B)
- **Annotator Failure Ratio**: Case B / (Case A + Case B)
- **Cases**: Breakdown of A, B, C to distinguish annotation vs. collection gaps

In [10]:
# Build annotation quality metrics
# For each level/target, decompose into case A (present + hateful), 
# B (present + non-hateful), and C (absent from union)

annotation_quality_data = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    # Case A: Posts with forms from this level/target labeled as hateful
    case_a = len(group[group['binary_hate'] == 1])
    
    # Case B: Posts with forms from this level/target labeled as non-hateful
    case_b = len(group[group['binary_hate'] == 0])
    
    # Case C: Posts that target this group but DON'T contain any forms 
    # from this level/target combination
    # Start with all posts targeting this group
    target_group_posts = data[data['targets'].str.contains(target, case=False, na=False)]
    
    # Find posts that contain any form from this glossary subset
    glossary_subset = glossary[(glossary['taxonomy_level'] == level) & 
                                (glossary['target'] == target)]
    forms_for_level_target = set(glossary_subset['surface_form'].unique())
    
    # Posts in target group that have forms from this level/target
    posts_with_forms = group['text_dedup_key'].unique()
    
    # Case C: Target group posts without forms from this level/target=
    case_c = len(target_group_posts[~target_group_posts['text_dedup_key'].isin(posts_with_forms)])
    
    # Calculate metrics
    matches_with_label = case_a + case_b
    correct_labeling_rate = case_a / matches_with_label if matches_with_label > 0 else 0
    annotator_failure_ratio = case_b / matches_with_label if matches_with_label > 0 else 0
    
    annotation_quality_data.append({
        'taxonomy_level': level,
        'target': target,
        'case_a_present_hateful': case_a,
        'case_b_present_nonhateful': case_b,
        'case_c_absent': case_c,
        'correct_labeling_rate': correct_labeling_rate,
        'annotator_failure_ratio': annotator_failure_ratio,
        'total_matches': matches_with_label,
        'total_target_group_posts': len(target_group_posts)
    })

annotation_quality_report = pd.DataFrame(annotation_quality_data)
annotation_quality_report

,taxonomy_level,target,case_a_present_hateful,case_b_present_nonhateful,case_c_absent,correct_labeling_rate,annotator_failure_ratio,total_matches,total_target_group_posts
0,2,african,0,2,4468,0.000000,1.000000,2,4468
1,3,african,1,34,4453,0.028571,0.971429,35,4468
2,3,hispanic,0,1,711,0.000000,1.000000,1,711
3,4,transgender women,1,2,1476,0.333333,0.666667,3,1476


In [11]:
# Analyze case breakdown: B/(B+C) indicates whether gap is annotation vs. collection
case_breakdown = annotation_quality_report.copy()
case_breakdown['case_b_c_ratio'] = (
    case_breakdown['case_b_present_nonhateful'] / 
    (case_breakdown['case_b_present_nonhateful'] + case_breakdown['case_c_absent'])
)
case_breakdown['collection_gap_prop'] = (
    case_breakdown['case_c_absent'] / 
    (case_breakdown['case_b_present_nonhateful'] + case_breakdown['case_c_absent'])
)

print("=" * 100)
print("ANNOTATION QUALITY AUDIT: CASE BREAKDOWN")
print("=" * 100)
print("\nCase B/C Ratio: Case B / (Case B + Case C)")
print("  - Closer to 1.0: primarily annotation failure (forms are there, missed by annotators)")
print("  - Closer to 0.0: primarily collection failure (forms not in benchmark at all)")
print("\n")
print(case_breakdown[['taxonomy_level', 'target', 'case_a_present_hateful', 
                       'case_b_present_nonhateful', 'case_c_absent', 
                       'case_b_c_ratio', 'collection_gap_prop']].to_string(index=False))

print("\n" + "=" * 100)
print("INTERPRETATION SUMMARY")
print("=" * 100)
for idx, row in case_breakdown.iterrows():
    level = row['taxonomy_level']
    target = row['target']
    ratio = row['case_b_c_ratio']
    b = row['case_b_present_nonhateful']
    c = row['case_c_absent']
    
    if b + c > 0:
        if ratio > 0.7:
            problem = "ANNOTATION FAILURE (primary issue is mislabeling)"
        elif ratio < 0.3:
            problem = "COLLECTION FAILURE (primary issue is missing forms)"
        else:
            problem = "MIXED (both annotation and collection issues)"
    else:
        problem = "INSUFFICIENT DATA"
    
    print(f"\nLevel {level}, {target.upper()}: {problem}")
    print(f"  Case A (correct): {row['case_a_present_hateful']} | " +
          f"Case B (missed): {b} | Case C (absent): {c}")

ANNOTATION QUALITY AUDIT: CASE BREAKDOWN

Case B/C Ratio: Case B / (Case B + Case C)
  - Closer to 1.0: primarily annotation failure (forms are there, missed by annotators)
  - Closer to 0.0: primarily collection failure (forms not in benchmark at all)




 taxonomy_level            target  case_a_present_hateful  case_b_present_nonhateful  case_c_absent  case_b_c_ratio  collection_gap_prop
              2           african                       0                          2           4468        0.000447             0.999553
              3           african                       1                         34           4453        0.007577             0.992423
              3          hispanic                       0                          1            711        0.001404             0.998596
              4 transgender women                       1                          2           1476        0.001353             0.998647

INTERPRETATION SUMMARY

Level 2, AFRICAN: COLLECTION FAILURE (primary issue is missing forms)
  Case A (correct): 0 | Case B (missed): 2 | Case C (absent): 4468

Level 3, AFRICAN: COLLECTION FAILURE (primary issue is missing forms)
  Case A (correct): 1 | Case B (missed): 34 | Case C (absent): 4453

Level 3, HISP

In [12]:
# Export annotation quality audit results
annotation_quality_path = cfg.output_dir / 'annotation_quality.tsv'
case_breakdown_path = cfg.output_dir / 'case_breakdown.tsv'

annotation_quality_report.to_csv(annotation_quality_path, sep='\t', index=False)
case_breakdown.to_csv(case_breakdown_path, sep='\t', index=False)

print("\nAnnotation quality audit exported:")
print(f"  - {annotation_quality_path}")
print(f"  - {case_breakdown_path}")


Annotation quality audit exported:
  - \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\coverage_audits\annotation_quality.tsv
  - \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\coverage_audits\case_breakdown.tsv


In [13]:
# Detailed breakdown by surface form: which forms are mislabeled (case B)?
form_labeling_detail = []

for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
    for form in group['found_forms'].unique():
        form_data = group[group['found_forms'] == form]
        
        hateful_count = len(form_data[form_data['binary_hate'] == 1])
        nonhateful_count = len(form_data[form_data['binary_hate'] == 0])
        
        if hateful_count + nonhateful_count > 0:
            labeling_accuracy = hateful_count / (hateful_count + nonhateful_count)
        else:
            labeling_accuracy = 0
        
        form_labeling_detail.append({
            'taxonomy_level': level,
            'target': target,
            'surface_form': form,
            'case_a_count': hateful_count,
            'case_b_count': nonhateful_count,
            'labeling_accuracy': labeling_accuracy,
            'type': form_data['type'].iloc[0] if len(form_data) > 0 else 'unknown'
        })

form_labeling_report = pd.DataFrame(form_labeling_detail).sort_values(
    by=['taxonomy_level', 'target', 'case_b_count'], ascending=[True, True, False]
)

print("\n" + "=" * 100)
print("SURFACE FORM LABELING DETAILS (Case A vs Case B)")
print("=" * 100)
print("\nForms sorted by mislabeling frequency (Case B count):")
print(form_labeling_report.to_string(index=False))

# Export form-level details
form_labeling_path = cfg.output_dir / 'form_labeling_detail.tsv'
form_labeling_report.to_csv(form_labeling_path, sep='\t', index=False)
print(f"\nForm-level labeling details exported to: {form_labeling_path}")


SURFACE FORM LABELING DETAILS (Case A vs Case B)

Forms sorted by mislabeling frequency (Case B count):
 taxonomy_level            target               surface_form  case_a_count  case_b_count  labeling_accuracy                                type
              2           african            lack of fathers             0             1           0.000000 stereotype-based target group label
              2           african            absentee father             0             1           0.000000 stereotype-based target group label
              3           african         affirmative action             1            34           0.028571                    concept (policy)
              3          hispanic end birthright citizenship             0             1           0.000000                    concept (policy)
              4 transgender women               actual women             1             1           0.500000   persona signal (self-referential)
              4 transgender wom

## Disparity Analysis

Assess disparate impact under equalized odds framework by comparing coverage and annotation quality metrics across target groups within the same coding level.

**Coverage Disparity**: Compare presence rates and type coverage between groups at the same level
**Annotation Disparity**: Compare correct labeling rates and annotator failure ratios between groups at the same level

In [14]:
# Coverage Disparity Analysis: Compare metrics across target groups within same level
coverage_disparities = []

for level in final_report['taxonomy_level'].unique():
    level_data = final_report[final_report['taxonomy_level'] == level]
    targets = level_data['target'].tolist()

    if len(targets) > 1:
        # Compare each pair of targets within this level
        for i in range(len(targets)):
            for j in range(i+1, len(targets)):
                target_a = targets[i]
                target_b = targets[j]

                data_a = level_data[level_data['target'] == target_a].iloc[0]
                data_b = level_data[level_data['target'] == target_b].iloc[0]

                # Presence rate disparity
                presence_gap = data_a['presence_rate'] - data_b['presence_rate']
                presence_gap_abs = abs(presence_gap)

                # Type coverage disparity
                type_gap = data_a['type_coverage'] - data_b['type_coverage']
                type_gap_abs = abs(type_gap)

                # Token frequency disparity (relative)
                if data_b['token_frequency'] > 0:
                    token_ratio = data_a['token_frequency'] / data_b['token_frequency']
                else:
                    token_ratio = float('inf') if data_a['token_frequency'] > 0 else 1.0

                coverage_disparities.append({
                    'taxonomy_level': level,
                    'target_a': target_a,
                    'target_b': target_b,
                    'presence_rate_a': data_a['presence_rate'],
                    'presence_rate_b': data_b['presence_rate'],
                    'presence_rate_gap': presence_gap,
                    'presence_rate_gap_abs': presence_gap_abs,
                    'type_coverage_a': data_a['type_coverage'],
                    'type_coverage_b': data_b['type_coverage'],
                    'type_coverage_gap': type_gap,
                    'type_coverage_gap_abs': type_gap_abs,
                    'token_freq_a': data_a['token_frequency'],
                    'token_freq_b': data_b['token_frequency'],
                    'token_freq_ratio': token_ratio
                })

coverage_disparity_report = pd.DataFrame(coverage_disparities)

print("=" * 120)
print("COVERAGE DISPARITY ANALYSIS: Within-Level Comparisons")
print("=" * 120)
print("\nPresence Rate Disparities (higher = target_a has better coverage):")
presence_cols = ['taxonomy_level', 'target_a', 'target_b', 'presence_rate_a', 'presence_rate_b', 'presence_rate_gap', 'presence_rate_gap_abs']
print(coverage_disparity_report[presence_cols].to_string(index=False))

print("\nType Coverage Disparities (higher = target_a has better type representation):")
type_cols = ['taxonomy_level', 'target_a', 'target_b', 'type_coverage_a', 'type_coverage_b', 'type_coverage_gap', 'type_coverage_gap_abs']
print(coverage_disparity_report[type_cols].to_string(index=False))

print("\nToken Frequency Ratios (ratio > 1 = target_a appears more frequently):")
token_cols = ['taxonomy_level', 'target_a', 'target_b', 'token_freq_a', 'token_freq_b', 'token_freq_ratio']
print(coverage_disparity_report[token_cols].to_string(index=False))

COVERAGE DISPARITY ANALYSIS: Within-Level Comparisons

Presence Rate Disparities (higher = target_a has better coverage):
 taxonomy_level target_a target_b  presence_rate_a  presence_rate_b  presence_rate_gap  presence_rate_gap_abs
              3  african hispanic              1.0         0.333333           0.666667               0.666667

Type Coverage Disparities (higher = target_a has better type representation):
 taxonomy_level target_a target_b  type_coverage_a  type_coverage_b  type_coverage_gap  type_coverage_gap_abs
              3  african hispanic              1.0              1.0                0.0                    0.0

Token Frequency Ratios (ratio > 1 = target_a appears more frequently):
 taxonomy_level target_a target_b  token_freq_a  token_freq_b  token_freq_ratio
              3  african hispanic            42             1              42.0


In [15]:
# Annotation Quality Disparity Analysis: Compare labeling metrics across target groups within same level
annotation_disparities = []

for level in annotation_quality_report['taxonomy_level'].unique():
    level_data = annotation_quality_report[annotation_quality_report['taxonomy_level'] == level]
    targets = level_data['target'].tolist()

    if len(targets) > 1:
        # Compare each pair of targets within this level
        for i in range(len(targets)):
            for j in range(i+1, len(targets)):
                target_a = targets[i]
                target_b = targets[j]

                data_a = level_data[level_data['target'] == target_a].iloc[0]
                data_b = level_data[level_data['target'] == target_b].iloc[0]

                # Correct labeling rate disparity
                labeling_gap = data_a['correct_labeling_rate'] - data_b['correct_labeling_rate']
                labeling_gap_abs = abs(labeling_gap)

                # Annotator failure ratio disparity
                failure_gap = data_a['annotator_failure_ratio'] - data_b['annotator_failure_ratio']
                failure_gap_abs = abs(failure_gap)

                # Case B/C ratio disparity (annotation vs collection problem attribution)
                b_c_gap = data_a['annotator_failure_ratio'] - data_b['annotator_failure_ratio']
                b_c_gap_abs = abs(b_c_gap)

                annotation_disparities.append({
                    'taxonomy_level': level,
                    'target_a': target_a,
                    'target_b': target_b,
                    'correct_labeling_rate_a': data_a['correct_labeling_rate'],
                    'correct_labeling_rate_b': data_b['correct_labeling_rate'],
                    'labeling_rate_gap': labeling_gap,
                    'labeling_rate_gap_abs': labeling_gap_abs,
                    'annotator_failure_ratio_a': data_a['annotator_failure_ratio'],
                    'annotator_failure_ratio_b': data_b['annotator_failure_ratio'],
                    'failure_ratio_gap': failure_gap,
                    'failure_ratio_gap_abs': failure_gap_abs,
                    'case_a_a': data_a['case_a_present_hateful'],
                    'case_a_b': data_b['case_a_present_hateful'],
                    'case_b_a': data_a['case_b_present_nonhateful'],
                    'case_b_b': data_b['case_b_present_nonhateful']
                })

annotation_disparity_report = pd.DataFrame(annotation_disparities)

print("\n" + "=" * 120)
print("ANNOTATION QUALITY DISPARITY ANALYSIS: Within-Level Comparisons")
print("=" * 120)
print("\nCorrect Labeling Rate Disparities (higher = target_a has better annotation quality):")
labeling_cols = ['taxonomy_level', 'target_a', 'target_b', 'correct_labeling_rate_a', 'correct_labeling_rate_b', 'labeling_rate_gap', 'labeling_rate_gap_abs']
print(annotation_disparity_report[labeling_cols].to_string(index=False))

print("\nAnnotator Failure Ratio Disparities (higher = target_a has worse annotation quality):")
failure_cols = ['taxonomy_level', 'target_a', 'target_b', 'annotator_failure_ratio_a', 'annotator_failure_ratio_b', 'failure_ratio_gap', 'failure_ratio_gap_abs']
print(annotation_disparity_report[failure_cols].to_string(index=False))

print("\nCase Counts for Context:")
case_cols = ['taxonomy_level', 'target_a', 'target_b', 'case_a_a', 'case_b_a', 'case_a_b', 'case_b_b']
print(annotation_disparity_report[case_cols].to_string(index=False))


ANNOTATION QUALITY DISPARITY ANALYSIS: Within-Level Comparisons

Correct Labeling Rate Disparities (higher = target_a has better annotation quality):
 taxonomy_level target_a target_b  correct_labeling_rate_a  correct_labeling_rate_b  labeling_rate_gap  labeling_rate_gap_abs
              3  african hispanic                 0.028571                      0.0           0.028571               0.028571

Annotator Failure Ratio Disparities (higher = target_a has worse annotation quality):
 taxonomy_level target_a target_b  annotator_failure_ratio_a  annotator_failure_ratio_b  failure_ratio_gap  failure_ratio_gap_abs
              3  african hispanic                   0.971429                        1.0          -0.028571               0.028571

Case Counts for Context:
 taxonomy_level target_a target_b  case_a_a  case_b_a  case_a_b  case_b_b
              3  african hispanic         1        34         0         1


In [16]:
# Export disparity analysis results
if len(coverage_disparity_report) > 0:
    coverage_disparity_path = cfg.output_dir / 'coverage_disparity.tsv'
    coverage_disparity_report.to_csv(coverage_disparity_path, sep='\t', index=False)
    print(f"\nCoverage disparity analysis exported to: {coverage_disparity_path}")

if len(annotation_disparity_report) > 0:
    annotation_disparity_path = cfg.output_dir / 'annotation_disparity.tsv'
    annotation_disparity_report.to_csv(annotation_disparity_path, sep='\t', index=False)
    print(f"Annotation disparity analysis exported to: {annotation_disparity_path}")

print(f"\nAll disparity analysis results exported to: {cfg.output_dir}")


Coverage disparity analysis exported to: \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\coverage_audits\coverage_disparity.tsv
Annotation disparity analysis exported to: \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\coverage_audits\annotation_disparity.tsv

All disparity analysis results exported to: \Users\ryanf\Documents\GitHub\benchmarking_dogwhistles\outputs\coverage_audits
